## LIME Explanation Plot — Conceptual Clarity
What LIME answers

Why did the model make this particular prediction for this particular instance?

It works by:

Sampling points around one instance

Perturbing features locally

Fitting a simple interpretable model (usually linear)

Explaining the prediction locally

📌 LIME is local-only (never global).

How to read a LIME plot
Bars

Right (green) → feature pushes prediction up

Left (red) → feature pushes prediction down

Length

Longer bar = stronger local influence

Important

Explanation is only valid near that instance

Different instance ⇒ different explanation

🧪 LIME EXPLANATION — ONE-CELL MASTER CODE

✔ Binary classification

✔ Tabular data

✔ Bar plot + notebook visualization

✔ Clean & reproducible

In [ ]:
# ============================================================
# LIME Explanation Plot — FULL ONE-CELL SCRIPT
# ============================================================

# (1) Install LIME (run once if needed)
! pip -q install lime

In [ ]:
# (2) Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from lime.lime_tabular import LimeTabularExplainer

np.random.seed(42)

# ------------------------------------------------------------
# (3) Dataset
# ------------------------------------------------------------
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# ------------------------------------------------------------
# (4) Train Model
# ------------------------------------------------------------
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# ------------------------------------------------------------
# (5) Initialize LIME Explainer
# ------------------------------------------------------------
explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X.columns.tolist(),
    class_names=["Benign", "Malignant"],
    mode="classification",
    discretize_continuous=True
)

# ------------------------------------------------------------
# (6) Select One Instance to Explain
# ------------------------------------------------------------
idx = 5   # change this to explain a different sample
instance = X_test.iloc[idx].values

# ------------------------------------------------------------
# (7) Generate LIME Explanation
# ------------------------------------------------------------
exp = explainer.explain_instance(
    data_row=instance,
    predict_fn=model.predict_proba,
    num_features=10
)

# ------------------------------------------------------------
# (8) LIME Explanation — Bar Plot (Matplotlib)
# ------------------------------------------------------------
fig = exp.as_pyplot_figure()
plt.title("LIME Local Explanation (Single Instance)")
plt.show()

# ------------------------------------------------------------
# (9) LIME Explanation — Textual Weights (Audit-friendly)
# ------------------------------------------------------------
print("LIME feature contributions:")
for feature, weight in exp.as_list():
    print(f"{feature:40s} : {weight:+.4f}")

# ------------------------------------------------------------
# (10) Optional: Display LIME Explanation Inline (Notebook)
# ------------------------------------------------------------
# Uncomment if running in Jupyter / Colab
# exp.show_in_notebook(show_table=True)
